# 3.1 Table One

**Class 3:** describe the study population, comparing people **with** and **without diabetes**. "Table 1" is the first table in almost every clinical research paper.

| Variable type | We report | Test for a difference |
|---|---|---|
| Categorical (gender, race…) | count (%) | chi-square |
| Continuous (age, BMI…) | mean (SD) | t-test |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

In [ ]:
fa <- prepare_analysis_data()
fa %>% count(diabetes)

## Categorical variables: count (%) and chi-square test

For each variable we count people in each category by diabetes status, add column percentages, and test whether the distribution differs with `chisq.test()`.

In [ ]:
summarise_categorical <- function(data, var) {
  tab <- table(data[[var]], data$diabetes)
  p <- suppressWarnings(chisq.test(tab)$p.value)
  as.data.frame.matrix(tab) %>%
    rownames_to_column("level") %>%
    mutate(
      `No diabetes`  = sprintf("%d (%.1f%%)", No,  100 * No  / sum(No)),
      `Diabetes`     = sprintf("%d (%.1f%%)", Yes, 100 * Yes / sum(Yes)),
      variable = var,
      p_value  = c(format.pval(p, digits = 2, eps = 0.001), rep("", n() - 1))
    ) %>%
    select(variable, level, `No diabetes`, `Diabetes`, p_value, n_no = No, n_yes = Yes)
}

categorical_vars <- c("gender", "race", "ethnicity", "age_group", "ptsd_doctor", "ptsd_treatment")
table_cat <- map_dfr(categorical_vars, ~ summarise_categorical(fa, .x))
table_cat %>% select(-n_no, -n_yes)

## Continuous variables: mean (SD) and t-test

`t.test()` compares the two group means (Welch's t-test, which doesn't assume equal variances).

In [ ]:
summarise_continuous <- function(data, var) {
  x_no  <- data[[var]][data$diabetes == "No"]
  x_yes <- data[[var]][data$diabetes == "Yes"]
  tibble(
    variable      = var,
    level         = "mean (SD)",
    `No diabetes` = sprintf("%.1f (%.1f)", mean(x_no),  sd(x_no)),
    `Diabetes`    = sprintf("%.1f (%.1f)", mean(x_yes), sd(x_yes)),
    p_value       = format.pval(t.test(x_yes, x_no)$p.value, digits = 2, eps = 0.001)
  )
}

continuous_vars <- c("age", "BMI", "Systolic_BP", "LDL_Cholesterol", "HbA1c")
table_cont <- map_dfr(continuous_vars, ~ summarise_continuous(fa, .x))
table_cont

## Put it together

In [ ]:
table_one <- bind_rows(
  tibble(variable = "N", level = "",
         `No diabetes` = as.character(sum(fa$diabetes == "No")),
         `Diabetes`    = as.character(sum(fa$diabetes == "Yes")), p_value = ""),
  table_cont,
  table_cat %>% select(-n_no, -n_yes)
)
table_one

## Before you share: check for small counts

The **Data and Statistics Dissemination Policy** doesn't allow sharing counts of **1 to 20**, or anything they could be calculated from (like a percentage of a small group). This flags any cell that's too small:

In [ ]:
table_cat %>%
  filter(between(n_no, 1, 20) | between(n_yes, 1, 20)) %>%
  select(variable, level, n_no, n_yes)

**Rows listed here can't be shared as they are.** Combine small categories (e.g. group races), or report them as *"< 20"*.

## Interpret

- Which variables differ between people with and without diabetes?
- Is a statistically significant difference always clinically important? (Look at the size of the BMI or HbA1c difference.)
- Why do we expect **HbA1c** to differ? (It's used to *diagnose* diabetes.)

> **Optional:** packages like `gtsummary` or `tableone` build publication-ready tables in one line. Install with `install.packages("gtsummary")`.